# 04 · Model Training
**Ninja Van Capstone 2026 — Last-Mile Delivery Failure Prediction**

This notebook trains three models:
| Model | Library | Tuning Method |
|---|---|---|
| LightGBM | `lightgbm` | Optuna (50 trials) |
| Random Forest | `sklearn` | GridSearchCV |
| Logistic Regression | `sklearn` | GridSearchCV |

Each model is tuned via 5-fold stratified cross-validation optimising **F1 on the failure class**.

**Outputs:**
- `/content/outputs/model_lgbm.joblib`
- `/content/outputs/model_rf.joblib`
- `/content/outputs/model_lr.joblib`
- `/content/outputs/cv_results.json`

> **Run order:** Run after `03_feature_engineering.ipynb`.


## 1 · Setup & Load Data

In [1]:
import sys, json, joblib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Locate repo root portably — works from notebooks/ in VS Code or Colab
REPO_DIR = Path.cwd()
while not (REPO_DIR / "config" / "config.yaml").exists():
    parent = REPO_DIR.parent
    if parent == REPO_DIR:
        raise RuntimeError("Cannot find repo root — run the notebook from inside the repo")
    REPO_DIR = parent

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROC_DIR = REPO_DIR / "data" / "processed"
OUT_DIR  = REPO_DIR / "outputs"
OUT_DIR.mkdir(exist_ok=True)

with open(REPO_DIR / "config" / "config.yaml") as f:
    cfg = yaml.safe_load(f)

RANDOM_STATE       = cfg["data"]["random_state"]
USE_SMOTE          = cfg["features"]["use_smote"]
OPTUNA_TRIALS      = cfg["models"]["lgbm"]["optuna_trials"]
LGBM_N_JOBS        = cfg["models"]["lgbm"]["n_jobs"]
TARGET_COL         = cfg["features"]["target_col"]
SEARCH_SAMPLE_SIZE = cfg["models"]["search_sample_size"]
CV_FOLDS           = cfg["models"]["cv_folds"]
CHECKPOINT_DIR     = REPO_DIR / cfg["outputs"]["checkpoint_dir"]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Config: random_state={RANDOM_STATE}, use_smote={USE_SMOTE}, "
      f"optuna_trials={OPTUNA_TRIALS}, lgbm_n_jobs={LGBM_N_JOBS}, "
      f"search_sample={SEARCH_SAMPLE_SIZE:,}, cv_folds={CV_FOLDS}")
print(f"REPO_DIR       : {REPO_DIR}")
print(f"PROC_DIR       : {PROC_DIR}")
print(f"OUT_DIR        : {OUT_DIR}")
print(f"CHECKPOINT_DIR : {CHECKPOINT_DIR}")
print("Device         : CPU")

Config: random_state=42, use_smote=False, optuna_trials=30, lgbm_n_jobs=4, search_sample=300,000, cv_folds=3
REPO_DIR       : /home/tashrif/ninja-van/last-mile-delivery-failure-prediction/nv-lastmile-training
PROC_DIR       : /home/tashrif/ninja-van/last-mile-delivery-failure-prediction/nv-lastmile-training/data/processed
OUT_DIR        : /home/tashrif/ninja-van/last-mile-delivery-failure-prediction/nv-lastmile-training/outputs
CHECKPOINT_DIR : /home/tashrif/ninja-van/last-mile-delivery-failure-prediction/nv-lastmile-training/outputs/checkpoints
Device         : CPU


In [2]:
# Load train split
train_df = pd.read_parquet(PROC_DIR / "features_train.parquet")

# If SMOTE was generated and enabled, use it
smote_path = PROC_DIR / "features_train_smote.parquet"
if USE_SMOTE and smote_path.exists():
    train_df = pd.read_parquet(smote_path)
    print(f"✅ Using SMOTE-balanced training set: {len(train_df):,} rows")
else:
    print(f"✅ Using original training set: {len(train_df):,} rows")

X_train_raw = train_df.drop(columns=[TARGET_COL])
y_train     = train_df[TARGET_COL].astype(int).values

# Load class weights
with open(OUT_DIR / "class_weights.json") as f:
    class_weights = json.load(f)
print(f"Class weights: {class_weights}")
print(f"Failure rate in train: {y_train.mean():.4f}")

✅ Using original training set: 3,611,728 rows
Class weights: {'0': 1.0, '1': 10.1478}
Failure rate in train: 0.0897


In [3]:
from features import get_preprocessor

try:
    preprocessor = joblib.load(OUT_DIR / "preprocessor.joblib")
    X_train = preprocessor.transform(X_train_raw)
    print("✅ Preprocessor loaded from disk")
except Exception as e:
    print(f"⚠️  Saved preprocessor incompatible ({type(e).__name__}: {e})")
    print("   Rebuilding preprocessor from training data...")
    preprocessor = get_preprocessor()
    X_train = preprocessor.fit_transform(X_train_raw)
    joblib.dump(preprocessor, OUT_DIR / "preprocessor.joblib", compress=3)
    print(f"✅ Preprocessor rebuilt and saved → {OUT_DIR / 'preprocessor.joblib'}")

with open(OUT_DIR / "feature_names.json") as f:
    feature_names = json.load(f)

print(f"Preprocessor ready — transformed shape: {X_train.shape}")
print(f"Feature count: {len(feature_names)}")

✅ Preprocessor loaded from disk
Preprocessor ready — transformed shape: (3611728, 20)
Feature count: 20


## 2 · LightGBM — Optuna Hyperparameter Search

> ⏱️ **Optimised:** 30 trials × 3-fold CV on a 300K-row stratified subsample, with LightGBM early stopping per fold. Estimated time: **3–6 min** on CPU, **1–2 min** on GPU. Best params are refit on the full training set.

In [ ]:
from train import run_optuna_lgbm

print("🔍 Starting Optuna search for LightGBM...")
print(f"   Trials: {OPTUNA_TRIALS}  |  CV folds: {CV_FOLDS}  |  "
      f"Search sample: {SEARCH_SAMPLE_SIZE:,}  |  Metric: F1 (failure class)")
print(f"   n_jobs: {LGBM_N_JOBS}  |  Checkpoint dir: {CHECKPOINT_DIR}")
print("-" * 60)

lgbm_best_params = run_optuna_lgbm(
    X_train, y_train,
    n_trials=OPTUNA_TRIALS,
    random_state=RANDOM_STATE,
    search_sample_size=SEARCH_SAMPLE_SIZE,
    n_splits=CV_FOLDS,
    n_jobs=LGBM_N_JOBS,
    checkpoint_dir=str(CHECKPOINT_DIR),
)

In [4]:
# Reload best params from checkpoint (run this cell to skip Optuna search above)
_params_path = CHECKPOINT_DIR / "lgbm_best_params.json"
with open(_params_path) as f:
    lgbm_best_params = json.load(f)
print(f"Loaded lgbm_best_params from {_params_path}")
print(lgbm_best_params)

Loaded lgbm_best_params from /home/tashrif/ninja-van/last-mile-delivery-failure-prediction/nv-lastmile-training/outputs/checkpoints/lgbm_best_params.json
{'learning_rate': 0.07371638852596306, 'num_leaves': 53, 'max_depth': 7, 'min_child_samples': 66, 'subsample': 0.7107668749360518, 'colsample_bytree': 0.8240130240094011, 'reg_alpha': 0.0014442684576132796, 'reg_lambda': 0.009691615639398445, 'scale_pos_weight': 6.881089593970588, 'n_estimators': 522}


In [6]:
# Cross-validate with best params to get reliable CV estimate
import lightgbm as lgb
from train import cross_validate_model

print("\n📊 Cross-validation with best LightGBM params:")
lgbm_cv_model = lgb.LGBMClassifier(
    **lgbm_best_params,
    random_state=RANDOM_STATE,
    objective="binary",
    metric="auc",
    verbosity=1,
    n_jobs=LGBM_N_JOBS,   # cap threads so 5 sequential folds don't exhaust memory
)
lgbm_cv_result = cross_validate_model(
    lgbm_cv_model, X_train, y_train,
    model_name="LightGBM",
    random_state=RANDOM_STATE,
    cv_n_jobs=1,           # run folds sequentially — model is already multi-threaded
)


📊 Cross-validation with best LightGBM params:
[LightGBM] [Info] Number of positive: 259188, number of negative: 2630194
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007233 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 259
[LightGBM] [Info] Number of data points in the train set: 2889382, number of used features: 2
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.089704 -> initscore=-2.317259
[LightGBM] [Info] Start training from score -2.317259
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warni

In [8]:
from train import train_lgbm

# Refit on full training set
print("\n🏋️ Fitting LightGBM on full training set...")
lgbm_model = train_lgbm(
    X_train, y_train,
    lgbm_best_params,
    checkpoint_path=str(CHECKPOINT_DIR / "lgbm_model.joblib"),
)

# Sanity check: training AUC
from sklearn.metrics import roc_auc_score
train_auc = roc_auc_score(y_train, lgbm_model.predict_proba(X_train)[:, 1])
print(f"   Training AUC-ROC (sanity): {train_auc:.4f}")

# Save to outputs (separate from checkpoint — this is the final artefact)
lgbm_path = OUT_DIR / "model_lgbm.joblib"
joblib.dump(lgbm_model, lgbm_path, compress=3)
print(f"✅ LightGBM saved → {lgbm_path}")


🏋️ Fitting LightGBM on full training set...
[LightGBM] [Info] Number of positive: 323985, number of negative: 3287743
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008789 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 259
[LightGBM] [Info] Number of data points in the train set: 3611728, number of used features: 2
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.089704 -> initscore=-2.317259
[LightGBM] [Info] Start training from score -2.317259
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning

## 3 · Random Forest — GridSearchCV

In [4]:
from train import grid_search_rf

_rf_params_path = CHECKPOINT_DIR / "rf_best_params.json"
_rf_ckpt_path   = CHECKPOINT_DIR / "rf_model.joblib"

if _rf_params_path.exists() and _rf_ckpt_path.exists():
    with open(_rf_params_path) as f:
        rf_best_params = json.load(f)
    rf_model = joblib.load(_rf_ckpt_path)
    print(f"✅ Loaded RF checkpoint — skipping grid search")
    print(f"   Best params: {rf_best_params}")
else:
    rf_param_grid = cfg["models"]["rf"]["param_grid"]
    rf_param_grid["max_depth"] = [
        None if v == "null" else v for v in rf_param_grid["max_depth"]
    ]

    print("🔍 GridSearch for Random Forest...")
    print(f"   Grid: {rf_param_grid}")
    print(f"   CV folds: {CV_FOLDS}  |  Search sample: {SEARCH_SAMPLE_SIZE:,}")
    print("-" * 60)

    rf_model, rf_best_params = grid_search_rf(
        X_train, y_train,
        param_grid=rf_param_grid,
        random_state=RANDOM_STATE,
        search_sample_size=SEARCH_SAMPLE_SIZE,
        n_splits=CV_FOLDS,
        model_n_jobs=4,   # threads per RF model
        gs_n_jobs=1,      # sequential grid search — avoids multiplying memory
    )

    with open(_rf_params_path, "w") as f:
        json.dump(rf_best_params, f, indent=2, default=str)
    joblib.dump(rf_model, _rf_ckpt_path, compress=3)
    print(f"   Saved RF checkpoint    → {_rf_ckpt_path}")
    print(f"   Saved RF best params   → {_rf_params_path}")

🔍 GridSearch for Random Forest...
   Grid: {'n_estimators': [100, 200], 'max_depth': [8, 15, None], 'class_weight': ['balanced']}
   CV folds: 3  |  Search sample: 300,000
------------------------------------------------------------
   Subsampled 300,000 rows for search (from 3,611,728 total, failure rate=0.090)
Fitting 3 folds for each of 6 candidates, totalling 18 fits

   GridSearch RF — best F1: 0.2831
   Best params: {'class_weight': 'balanced', 'max_depth': None, 'n_estimators': 200}
param_class_weight param_max_depth  param_n_estimators  mean_test_score  std_test_score  rank_test_score
          balanced            None                 200         0.283097        0.001797                1
          balanced            None                 100         0.282584        0.001619                2
          balanced              15                 100         0.269302        0.002131                3
          balanced              15                 200         0.268279        0.0016

In [ ]:
# Reload RF checkpoint (run this cell to skip grid search above)
_rf_params_path = CHECKPOINT_DIR / "rf_best_params.json"
_rf_ckpt_path   = CHECKPOINT_DIR / "rf_model.joblib"
with open(_rf_params_path) as f:
    rf_best_params = json.load(f)
rf_model = joblib.load(_rf_ckpt_path)
print(f"Loaded rf_best_params from {_rf_params_path}")
print(rf_best_params)

In [ ]:
# CV result with best params
from sklearn.ensemble import RandomForestClassifier
from train import cross_validate_model

print("\n📊 Cross-validation with best RF params:")
rf_cv_model = RandomForestClassifier(
    **rf_best_params,
    n_jobs=4,              # cap threads — folds run sequentially below
    random_state=RANDOM_STATE,
)
rf_cv_result = cross_validate_model(
    rf_cv_model, X_train, y_train,
    model_name="Random Forest",
    random_state=RANDOM_STATE,
    cv_n_jobs=1,           # sequential folds to avoid OOM on full 3.6M dataset
)



📊 Cross-validation with best RF params:
   Random Forest              AUC=0.7630±0.0006  F1=0.2928±0.0005


NameError: name 'roc_auc_score' is not defined

In [6]:
# Training AUC sanity check
from sklearn.metrics import roc_auc_score
rf_train_auc = roc_auc_score(y_train, rf_model.predict_proba(X_train)[:, 1])
print(f"   Training AUC-ROC (sanity): {rf_train_auc:.4f}")

# Save final artefact
rf_path = OUT_DIR / "model_rf.joblib"
joblib.dump(rf_model, rf_path, compress=3)
print(f"✅ Random Forest saved → {rf_path}")

   Training AUC-ROC (sanity): 0.7657
✅ Random Forest saved → /home/tashrif/ninja-van/last-mile-delivery-failure-prediction/nv-lastmile-training/outputs/model_rf.joblib


## 4 · Logistic Regression — GridSearchCV

In [ ]:
from train import grid_search_lr

lr_param_grid = cfg["models"]["lr"]["param_grid"]

print("🔍 GridSearch for Logistic Regression...")
print(f"   Grid: {lr_param_grid}")
print(f"   CV folds: {CV_FOLDS}  |  Search sample: {SEARCH_SAMPLE_SIZE:,}")
print("-" * 60)

lr_model, lr_best_params = grid_search_lr(
    X_train, y_train,
    param_grid=lr_param_grid,
    random_state=RANDOM_STATE,
    search_sample_size=SEARCH_SAMPLE_SIZE,
    n_splits=CV_FOLDS,
)

In [ ]:
from sklearn.linear_model import LogisticRegression

print("\n📊 Cross-validation with best LR params:")
lr_cv_model = LogisticRegression(**lr_best_params, solver="lbfgs",
                                   max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE)
lr_cv_result = cross_validate_model(lr_cv_model, X_train, y_train,
                                     model_name="Logistic Regression", random_state=RANDOM_STATE)

lr_train_auc = roc_auc_score(y_train, lr_model.predict_proba(X_train)[:, 1])
print(f"   Training AUC-ROC (sanity): {lr_train_auc:.4f}")

lr_path = OUT_DIR / "model_lr.joblib"
joblib.dump(lr_model, lr_path, compress=3)
print(f"✅ Logistic Regression saved → {lr_path}")

## 5 · CV Results Summary

In [7]:
cv_results = {
    "lgbm": lgbm_cv_result,
    "rf":   rf_cv_result,
    "lr":   lr_cv_result,
}

# Pretty-print comparison table
print("\n" + "=" * 65)
print("  CROSS-VALIDATION RESULTS SUMMARY (5-fold, optimised on F1)")
print("=" * 65)
print(f"  {'Model':20s}  {'AUC-ROC':>9s}  {'±':>5s}  {'F1':>7s}  {'±':>5s}")
print(f"  {'─'*20}  {'─'*9}  {'─'*5}  {'─'*7}  {'─'*5}")
for name, res in cv_results.items():
    print(f"  {res['model_name']:20s}  {res['mean_auc']:9.4f}  "
          f"{res['std_auc']:5.4f}  {res['mean_f1']:7.4f}  {res['std_f1']:5.4f}")
print("=" * 65)

# Bar chart: CV F1 comparison
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
names = [r["model_name"] for r in cv_results.values()]
aucs  = [r["mean_auc"]   for r in cv_results.values()]
f1s   = [r["mean_f1"]    for r in cv_results.values()]
auc_stds = [r["std_auc"] for r in cv_results.values()]
f1_stds  = [r["std_f1"]  for r in cv_results.values()]
colors   = ["#E63946", "#457B9D", "#2A9D8F"]

axes[0].bar(names, aucs, yerr=auc_stds, color=colors, edgecolor="white",
            capsize=4, alpha=0.85)
axes[0].set_ylim(0.5, 1.0)
axes[0].set_title("CV AUC-ROC (5-fold)", fontweight="bold")
axes[0].set_ylabel("AUC-ROC")
for i, (v, e) in enumerate(zip(aucs, auc_stds)):
    axes[0].text(i, v + e + 0.005, f"{v:.3f}", ha="center", fontsize=10)

axes[1].bar(names, f1s, yerr=f1_stds, color=colors, edgecolor="white",
            capsize=4, alpha=0.85)
axes[1].set_ylim(0.0, 1.0)
axes[1].set_title("CV F1 — Failure Class (5-fold)", fontweight="bold")
axes[1].set_ylabel("F1 Score")
for i, (v, e) in enumerate(zip(f1s, f1_stds)):
    axes[1].text(i, v + e + 0.005, f"{v:.3f}", ha="center", fontsize=10)

plt.suptitle("Cross-Validation Results Comparison", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(OUT_DIR / "cv_results_comparison.png", bbox_inches="tight", dpi=150)
plt.show()

# Save CV results JSON
with open(OUT_DIR / "cv_results.json", "w") as f:
    # Remove non-serialisable fields
    save_cv = {k: {kk: vv for kk, vv in v.items() if not isinstance(vv, np.ndarray)}
               for k, v in cv_results.items()}
    json.dump(save_cv, f, indent=2, default=float)
print(f"\n✅ CV results saved → {OUT_DIR / 'cv_results.json'}")

NameError: name 'lgbm_cv_result' is not defined

In [ ]:
print("=" * 70)
print("  NOTEBOOK 04 COMPLETE")
print("=" * 70)
print(f"  model_lgbm.joblib → {OUT_DIR / 'model_lgbm.joblib'}")
print(f"  model_rf.joblib   → {OUT_DIR / 'model_rf.joblib'}")
print(f"  model_lr.joblib   → {OUT_DIR / 'model_lr.joblib'}")
print("\n✅ Proceed to 05_model_evaluation.ipynb")